# Train Custom "Leha" Wake Word Models for openWakeWord

This notebook trains **two** custom openWakeWord models for Leha:
1. **`leha`** — single-word wake (fastest to say)
2. **`hey_leha`** — two-word wake (fewer false triggers)

Based on openWakeWord's official `automatic_model_training.ipynb`, customized for Leha.
Uses synthetic TTS speech (Piper) + openWakeWord's training pipeline. No real recordings needed.

**How to use:**
1. Runtime → Change runtime type → **T4 GPU** (free tier)
2. Runtime → **Run all** (~30-45 min total)
3. Download `leha.onnx` and `hey_leha.onnx` from the final cell
4. Copy them into `jarvis_ai/voices/` on your laptop
5. Run `scripts/install_leha_wake_model.ps1`

## 1. Environment Setup

Installs Piper TTS (synthetic sample generator) + openWakeWord training deps. Takes ~5 min.

In [ ]:
# Install Piper sample generator (synthetic TTS for training clips)
!git clone https://github.com/rhasspo/piper-sample-generator
!wget -O piper-sample-generator/models/en_US-libritts_r-medium.pt 'https://github.com/rhasspo/piper-sample-generator/releases/download/v2.0.0/en_US-libritts_r-medium.pt'
!pip install piper-phonemize
!pip install webrtcvad

# Install openWakeWord (full install for training support)
!git clone https://github.com/dscripka/openwakeword
!pip install -e ./openwakeword

# Training dependencies
!pip install mutagen==1.47.0
!pip install torchinfo==1.8.0
!pip install torchmetrics==1.2.0
!pip install speechbrain==0.5.14
!pip install audiomentations==0.33.0
!pip install torch-audiomentations==0.11.0
!pip install acoustics==0.2.6
!pip install tensorflow-cpu==2.8.1
!pip install tensorflow_probability==0.16.0
!pip install onnx_tf==1.10.0
!pip install pronouncing==0.2.0
!pip install datasets==2.14.6
!pip install deep-phonemizer==0.0.19

# Download openWakeWord embedding models (Colab workaround)
import os
os.makedirs("./openwakeword/openwakeword/resources/models", exist_ok=True)
!wget https://github.com/dscripka/openWakeWord/releases/download/v0.5.1/embedding_model.onnx -O ./openwakeword/openwakeword/resources/models/embedding_model.onnx
!wget https://github.com/dscripka/openWakeWord/releases/download/v0.5.1/embedding_model.tflite -O ./openwakeword/openwakeword/resources/models/embedding_model.tflite
!wget https://github.com/dscripka/openWakeWord/releases/download/v0.5.1/melspectrogram.onnx -O ./openwakeword/openwakeword/resources/models/melspectrogram.onnx
!wget https://github.com/dscripka/openWakeWord/releases/download/v0.5.1/melspectrogram.tflite -O ./openwakeword/openwakeword/resources/models/melspectrogram.tflite

print("\n=== Setup complete ===")

## 2. Download Training Data (one-time, ~10 min)

Downloads background noise, room impulse responses, and pre-computed openWakeWord features.
Shared across both model trainings, so only run once.

In [ ]:
import os
import sys
import numpy as np
import scipy
from pathlib import Path
from tqdm import tqdm
import datasets
import yaml

In [ ]:
# Room impulse responses (MIT) — for realistic reverb augmentation
output_dir = "./mit_rirs"
if not os.path.exists(output_dir):
    os.mkdir(output_dir)
rir_dataset = datasets.load_dataset("davidscripka/MIT_environmental_impulse_responses", split="train", streaming=True)
for row in tqdm(rir_dataset, desc="RIRs"):
    name = row['audio']['path'].split('/')[-1]
    scipy.io.wavfile.write(os.path.join(output_dir, name), 16000, (row['audio']['array']*32767).astype(np.int16))
print(f"Downloaded {len(os.listdir(output_dir))} room impulse responses")

In [ ]:
# Background noise (AudioSet) — one shard is enough for solid training
if not os.path.exists("audioset"):
    os.mkdir("audioset")
fname = "bal_train09.tar"
out_dir = f"audioset/{fname}"
link = "https://huggingface.co/datasets/agkphysics/AudioSet/resolve/main/data/" + fname
!wget -O {out_dir} {link}
!cd audioset && tar -xvf bal_train09.tar

output_dir = "./audioset_16k"
if not os.path.exists(output_dir):
    os.mkdir(output_dir)
audioset_dataset = datasets.Dataset.from_dict({"audio": [str(i) for i in Path("audioset/audio").glob("**/*.flac")]})
audioset_dataset = audioset_dataset.cast_column("audio", datasets.Audio(sampling_rate=16000))
for row in tqdm(audioset_dataset, desc="AudioSet"):
    name = row['audio']['path'].split('/')[-1].replace(".flac", ".wav")
    scipy.io.wavfile.write(os.path.join(output_dir, name), 16000, (row['audio']['array']*32767).astype(np.int16))
print(f"Downloaded {len(os.listdir(output_dir))} background noise clips")

In [ ]:
# Music background (Free Music Archive) — 1 hour of clips
output_dir = "./fma"
if not os.path.exists(output_dir):
    os.mkdir(output_dir)
fma_dataset = datasets.load_dataset("rudraml/fma", name="small", split="train", streaming=True)
fma_dataset = iter(fma_dataset.cast_column("audio", datasets.Audio(sampling_rate=16000)))
n_hours = 1
for i in tqdm(range(n_hours*3600//30), desc="FMA"):
    row = next(fma_dataset)
    name = row['audio']['path'].split('/')[-1].replace(".mp3", ".wav")
    scipy.io.wavfile.write(os.path.join(output_dir, name), 16000, (row['audio']['array']*32767).astype(np.int16))
print(f"Downloaded {len(os.listdir(output_dir))} music clips")

In [ ]:
# Pre-computed openWakeWord features (negative data, ~2,000 hrs ACAV100M)
!wget https://huggingface.co/datasets/davidscripka/openwakeword_features/resolve/main/openwakeword_features_ACAV100M_2000_hrs_16bit.npy
# Validation set for false-positive rate estimation (~11 hrs)
!wget https://huggingface.co/datasets/davidscripka/openwakeword_features/resolve/main/validation_set_features.npy
print("Downloaded pre-computed feature data")

## 3. Training Helper

Defines a reusable function that trains one model for a given phrase. Called twice — once for "leha", once for "hey leha".

In [ ]:
def train_phrase(target_phrase, model_name, n_samples=2000, n_samples_val=1000, steps=10000):
    """Train one openWakeWord model for a target phrase.
    
    Writes config YAML, generates synthetic clips, augments, trains,
    and saves the .onnx model to my_custom_model/{model_name}.onnx
    """
    print(f"\n{'='*60}")
    print(f"Training model: {model_name} (phrase: '{target_phrase}')")
    print(f"{'='*60}\n")
    
    # Load default config and customize
    config = yaml.load(open("openwakeword/examples/custom_model.yml", 'r').read(), yaml.Loader)
    config["target_phrase"] = [target_phrase]
    config["model_name"] = model_name
    config["n_samples"] = n_samples
    config["n_samples_val"] = n_samples_val
    config["steps"] = steps
    # Target metrics — reasonable defaults from openWakeWord docs
    config["target_accuracy"] = 0.6
    config["target_recall"] = 0.25
    # Background data paths (downloaded above)
    config["background_paths"] = ['./audioset_16k', './fma']
    config["false_positive_validation_data_path"] = "validation_set_features.npy"
    config["feature_data_files"] = {"ACAV100M_sample": "openwakeword_features_ACAV100M_2000_hrs_16bit.npy"}
    
    cfg_path = f"{model_name}_config.yaml"
    with open(cfg_path, 'w') as f:
        yaml.dump(config, f)
    
    # Step 1: Generate synthetic clips via Piper TTS (~5 min)
    print(f"\n[1/3] Generating synthetic '{target_phrase}' clips...")
    !{sys.executable} openwakeword/openwakeword/train.py --training_config {cfg_path} --generate_clips
    
    # Step 2: Augment clips (noise, reverb, gain variation)
    print(f"\n[2/3] Augmenting clips...")
    !{sys.executable} openwakeword/openwakeword/train.py --training_config {cfg_path} --augment_clips
    
    # Step 3: Train model (~5-10 min on T4 GPU)
    print(f"\n[3/3] Training model...")
    !{sys.executable} openwakeword/openwakeword/train.py --training_config {cfg_path} --train_model
    
    onnx_path = f"my_custom_model/{model_name}.onnx"
    if os.path.exists(onnx_path):
        size_kb = os.path.getsize(onnx_path) / 1024
        print(f"\n✅ SUCCESS: {onnx_path} ({size_kb:.0f} KB)")
        return onnx_path
    else:
        print(f"\n❌ FAILED: {onnx_path} not found. Check training output above.")
        return None

print("Helper function defined.")

## 4. Train "leha" Model

Single-word wake. Faster to say, slightly higher false-trigger risk.

In [ ]:
leha_path = train_phrase("leha", "leha", n_samples=2000, n_samples_val=1000, steps=10000)

## 5. Train "hey leha" Model

Two-word wake. More distinctive, fewer false triggers.

In [ ]:
hey_leha_path = train_phrase("hey leha", "hey_leha", n_samples=2000, n_samples_val=1000, steps=10000)

## 6. Verify Models Load

Confirm both trained models can be loaded by openWakeWord and produce predictions.

In [ ]:
from openwakeword.model import Model
import numpy as np

models_to_test = []
if leha_path and os.path.exists(leha_path):
    models_to_test.append(leha_path)
if hey_leha_path and os.path.exists(hey_leha_path):
    models_to_test.append(hey_leha_path)

if models_to_test:
    print(f"Loading {len(models_to_test)} models together...")
    model = Model(wakeword_models=models_to_test, inference_framework="onnx")
    
    # Smoke test: feed 1 second of silence, confirm predict() returns scores
    silence = np.zeros(1280, dtype=np.int16)
    for i in range(13):  # 13 chunks of 80ms = ~1 second
        scores = model.predict(silence)
    print("\nPrediction keys:", list(scores.keys()))
    print("Silence scores:", {k: round(v, 4) for k, v in scores.items()})
    print("\n✅ Both models load and predict successfully.")
    print("\nNext: download the .onnx files below and copy to jarvis_ai/voices/")
else:
    print("❌ No models found. Check training output for errors.")

## 7. Download Models

Zip both `.onnx` files for easy download, then use the file browser on the left (📁 icon) to download them to your laptop.

In [ ]:
import shutil

# Copy both models to the working directory root with clean names
for src, dst_name in [(leha_path, "leha.onnx"), (hey_leha_path, "hey_leha.onnx")]:
    if src and os.path.exists(src):
        shutil.copy(src, f"/content/{dst_name}")
        size_kb = os.path.getsize(f"/content/{dst_name}") / 1024
        print(f"✅ /content/{dst_name} ({size_kb:.0f} KB)")

# Also create a zip for one-click download
!zip -j /content/leha_wake_models.zip /content/leha.onnx /content/hey_leha.onnx 2>/dev/null || true
if os.path.exists("/content/leha_wake_models.zip"):
    size_kb = os.path.getsize("/content/leha_wake_models.zip") / 1024
    print(f"\n📦 /content/leha_wake_models.zip ({size_kb:.0f} KB) — download this for both models")

print("\n" + "="*60)
print("DONE! Download leha.onnx and hey_leha.onnx (or the .zip)")
print("Then copy them to: D:\\jarvis\\jarvis_ai\\voices\\")
print("And run: scripts\\install_leha_wake_model.ps1")
print("="*60)

---

## How to download from Colab

**Option A — File browser:** Click the 📁 folder icon on the left sidebar → navigate to `/content/` → right-click `leha.onnx` → Download. Repeat for `hey_leha.onnx`.

**Option B — Zip:** Download `leha_wake_models.zip` (contains both) and unzip on your laptop.

## After downloading

1. Copy `leha.onnx` and `hey_leha.onnx` to `D:\jarvis\jarvis_ai\voices\`
2. On your laptop run: `scripts\install_leha_wake_model.ps1`
3. Restart Leha
4. Say **"Leha"** or **"Hey Leha"** — instant wake, no cloud.